# MHA、MQA、GQA：从注意力等价到 KV 容量账本

**面试问题：Query head 与 KV head 怎样映射，输出和 KV 显存如何验证？**

## 回答主线

先明确业务目标和数据合同，再给出可比较的朴素基线；随后手写核心算法，展示中间状态、最终指标和失败路径。本 Notebook 的断言只出现在最后，用于保护关键不变量；学习重点是前面的输入、过程、对照与解释。

## 真实案例

一个 8 个 query head 的客服模型要在单卡上同时服务更多会话。团队考虑把 8 个 KV head 改成 2 个 GQA head 或 1 个 MQA head。案例手写 causal attention，先用共享后的 MHA 作为 full-reference，再验证 GQA 头映射逐值等价，最后把序列长度、层数、dtype 和并发带入 KV 容量表。

### 输入预览：一次四 token 解码与头配置

In [1]:
import numpy as np  # 导入矩阵运算以从零实现多头注意力。

tokens = ["<用户>", "退款", "多久到账", "?"]  # 给四个 token 附上可读客服语义。
query_heads = 8  # 配置八个 query head 模拟常见 GQA 模型比例。
kv_heads = 2  # 配置两个 KV head，使每四个 query head 共享一组 K/V。
head_dim = 4  # 使用四维教学 head 便于直接检查矩阵。
rng = np.random.default_rng(2304)  # 固定随机源保证注意力输出可复现。
queries = rng.normal(size=(len(tokens), query_heads, head_dim))  # 生成每个 token 的八个 query head。
keys_gqa = rng.normal(size=(len(tokens), kv_heads, head_dim))  # 生成两个共享 KV head 的 key。
values_gqa = rng.normal(size=(len(tokens), kv_heads, head_dim))  # 生成两个共享 KV head 的 value。
print("token 序列：", tokens)  # 展示注意力实际处理的语义输入。
print(f"Q shape={queries.shape}，K/V shape={keys_gqa.shape}，每个 KV head 服务 {query_heads // kv_heads} 个 query head")  # 展示 GQA 的核心张量合同。

token 序列： ['<用户>', '退款', '多久到账', '?']
Q shape=(4, 8, 4)，K/V shape=(4, 2, 4)，每个 KV head 服务 4 个 query head


## Baseline 基线：把共享 K/V 物理复制成八个 MHA head

In [2]:
def stable_softmax(scores):  # 实现数值稳定的最后一维 softmax。
    shifted = scores - np.max(scores, axis=-1, keepdims=True)  # 减去最大值避免指数溢出。
    exponentials = np.exp(shifted)  # 计算平移后的指数权重。
    return exponentials / exponentials.sum(axis=-1, keepdims=True)  # 返回归一化注意力概率。

def causal_head_attention(query, key, value):  # 实现单个 head 的完整 causal self-attention。
    scores = query @ key.T / np.sqrt(query.shape[-1])  # 计算缩放点积注意力分数。
    causal_mask = np.triu(np.ones_like(scores, dtype=bool), k=1)  # 构造禁止观察未来 token 的上三角 mask。
    scores = np.where(causal_mask, -1e9, scores)  # 把未来位置替换为近似负无穷。
    probabilities = stable_softmax(scores)  # 将合法历史位置转换为概率。
    return probabilities @ value, probabilities  # 返回加权输出和可解释的注意力矩阵。

keys_mha = np.repeat(keys_gqa, query_heads // kv_heads, axis=1)  # 把每个 GQA key 物理复制四次构造 MHA reference。
values_mha = np.repeat(values_gqa, query_heads // kv_heads, axis=1)  # 同样复制 value 构造完整八头缓存。
mha_outputs = np.stack([causal_head_attention(queries[:, head], keys_mha[:, head], values_mha[:, head])[0] for head in range(query_heads)], axis=1)  # 对八个物理 MHA head 分别计算输出。
print("MHA reference 最后 token 的八头输出均值：", np.round(mha_outputs[-1].mean(axis=0), 4))  # 展示 full-reference 的具体数值结果。
print(f"物理 MHA K/V 元素数={keys_mha.size + values_mha.size}")  # 展示复制共享头会产生的缓存浪费。

MHA reference 最后 token 的八头输出均值： [-0.121   0.0645 -0.5757  0.1731]
物理 MHA K/V 元素数=256


### 核心实现：GQA 逻辑映射而不复制缓存

In [3]:
def grouped_attention(all_queries, shared_keys, shared_values):  # 实现 query head 到 KV head 的连续分组映射。
    token_count, q_heads, dimension = all_queries.shape  # 读取输入张量的三个关键维度。
    shared_head_count = shared_keys.shape[1]  # 读取实际存储的 KV head 数量。
    group_size = q_heads // shared_head_count  # 计算每个 KV head 服务的 query head 数。
    outputs = np.zeros((token_count, q_heads, dimension))  # 分配与 MHA 输出相同形状的结果张量。
    last_token_weights = {}  # 保存最后 token 的注意力权重用于解释头映射。
    for query_head in range(q_heads):  # 逐个 query head 执行注意力。
        kv_head = query_head // group_size  # 用连续分组规则选择共享 KV head。
        output, probabilities = causal_head_attention(all_queries[:, query_head], shared_keys[:, kv_head], shared_values[:, kv_head])  # 在不复制 K/V 的情况下计算该头输出。
        outputs[:, query_head] = output  # 写入当前 query head 的完整序列输出。
        last_token_weights[query_head] = probabilities[-1]  # 保存最后 token 对历史位置的权重。
    return outputs, last_token_weights  # 返回 GQA 输出与可观察权重。

gqa_outputs, gqa_weights = grouped_attention(queries, keys_gqa, values_gqa)  # 运行真正只存两个 KV head 的 GQA。
equivalence_error = float(np.max(np.abs(gqa_outputs - mha_outputs)))  # 计算 GQA 与物理复制 reference 的最大误差。
print(f"GQA 与共享后 MHA reference 的最大误差={equivalence_error:.3e}")  # 展示逻辑共享不会改变数学输出。
for head in (0, 3, 4, 7):  # 选取两个分组的边界 head 展示映射和注意力。
    print(f"query_head={head} -> kv_head={head // 4}，最后 token 权重={np.round(gqa_weights[head], 3)}")  # 展示连续分组规则对应的历史关注位置。

GQA 与共享后 MHA reference 的最大误差=0.000e+00
query_head=0 -> kv_head=0，最后 token 权重=[0.101 0.238 0.342 0.319]
query_head=3 -> kv_head=0，最后 token 权重=[0.319 0.398 0.151 0.131]
query_head=4 -> kv_head=1，最后 token 权重=[0.124 0.689 0.145 0.042]
query_head=7 -> kv_head=1，最后 token 权重=[0.37  0.46  0.118 0.052]


## 结果解读：把 KV head 数转换成服务容量

In [4]:
def kv_cache_gib(layers, concurrent_sequences, sequence_length, heads, dimension, bytes_per_value=2):  # 计算指定并发下 K/V cache 的理论 GiB。
    values = 2 * layers * concurrent_sequences * sequence_length * heads * dimension  # 同时统计 K 和 V 两份缓存元素。
    return values * bytes_per_value / 1024 ** 3  # 把元素数量转换成 GiB。

capacity_rows = []  # 收集 MHA、GQA 和 MQA 的容量对比记录。
for name, heads in (("MHA", 8), ("GQA", 2), ("MQA", 1)):  # 分别计算三种 KV head 配置。
    gib = kv_cache_gib(layers=32, concurrent_sequences=64, sequence_length=4096, heads=heads, dimension=128)  # 使用 32 层、64 并发和 4K 上下文的实际规划参数。
    capacity_rows.append((name, heads, gib))  # 保存名称、head 数和容量结果。
print("模式  KV heads  64并发×4K 的 KV GiB")  # 输出容量对照表表头。
for name, heads, gib in capacity_rows:  # 逐行展示三种注意力缓存成本。
    print(f"{name:<4} {heads:>8} {gib:>18.2f}")  # 显示减少 KV head 如何近似线性节省显存。
print("解读：GQA 的主要收益是缓存容量和 decode 带宽；是否保持质量必须在目标 checkpoint 与任务上重新评测。")  # 区分数学容量结论与模型质量结论。

模式  KV heads  64并发×4K 的 KV GiB
MHA         8              32.00
GQA         2               8.00
MQA         1               4.00
解读：GQA 的主要收益是缓存容量和 decode 带宽；是否保持质量必须在目标 checkpoint 与任务上重新评测。


## 失败案例：用取模映射替代连续分组

In [5]:
def wrong_grouped_attention(all_queries, shared_keys, shared_values):  # 构造把 query head 交错映射到 KV head 的常见错误。
    outputs = np.zeros_like(all_queries)  # 分配错误实现的输出张量。
    for query_head in range(all_queries.shape[1]):  # 逐个 query head 计算错误映射结果。
        kv_head = query_head % shared_keys.shape[1]  # 错误地使用取模形成交错分组。
        outputs[:, query_head] = causal_head_attention(all_queries[:, query_head], shared_keys[:, kv_head], shared_values[:, kv_head])[0]  # 计算使用错误 KV head 的输出。
    return outputs  # 返回形状正确但语义错误的注意力结果。

wrong_outputs = wrong_grouped_attention(queries, keys_gqa, values_gqa)  # 运行错误映射路径。
wrong_error = float(np.max(np.abs(wrong_outputs - mha_outputs)))  # 对比 checkpoint 约定的连续分组 reference。
print(f"错误取模映射的最大输出误差={wrong_error:.4f}")  # 展示 shape 测试无法发现的严重数值错误。
print("正确映射前八个 query head：", [head // 4 for head in range(8)])  # 展示 checkpoint 使用的连续分组。
print("错误映射前八个 query head：", [head % 2 for head in range(8)])  # 展示看似均匀但与权重语义不兼容的交错分组。

错误取模映射的最大输出误差=2.6035
正确映射前八个 query head： [0, 0, 0, 0, 1, 1, 1, 1]
错误映射前八个 query head： [0, 1, 0, 1, 0, 1, 0, 1]


### 生产边界

In [6]:
cache_contract = {"attention_type": "GQA", "query_heads": query_heads, "kv_heads": kv_heads, "head_dim": head_dim, "group_rule": "contiguous", "rope_version": "rope-v2", "dtype": "fp16"}  # 构造缓存与 checkpoint 必须共同绑定的版本合同。
print("KV cache 合同：", cache_contract)  # 展示服务端不能只依赖张量 shape 推断语义。
print("生产替换点：真实实现还要验证 RoPE 位置、paged block、tensor parallel 分片、fused kernel 和 checkpoint 转换质量。")  # 明确 NumPy reference 与高性能内核之间的差距。

KV cache 合同： {'attention_type': 'GQA', 'query_heads': 8, 'kv_heads': 2, 'head_dim': 4, 'group_rule': 'contiguous', 'rope_version': 'rope-v2', 'dtype': 'fp16'}
生产替换点：真实实现还要验证 RoPE 位置、paged block、tensor parallel 分片、fused kernel 和 checkpoint 转换质量。


## 回归测试：只保护 causal、映射和容量关系

In [7]:
assert equivalence_error < 1e-12  # 验证连续分组 GQA 与物理复制 reference 数学等价。
assert np.allclose(causal_head_attention(queries[:, 0], keys_gqa[:, 0], values_gqa[:, 0])[1][0, 1:], [0.0, 0.0, 0.0])  # 验证首 token 的 causal attention 不会观察未来位置。
assert wrong_error > 1e-3  # 验证错误取模映射能被数值 oracle 稳定识别。
assert capacity_rows[1][2] == capacity_rows[0][2] / 4  # 验证两个 KV head 相对八头 MHA 节省四倍缓存。
assert cache_contract["kv_heads"] < cache_contract["query_heads"]  # 验证发布合同确实描述 GQA 而非 MHA。
print("回归测试通过：causal mask、头映射、容量比例和缓存合同均成立。")  # 用可见结果总结真正被保护的不变量。

回归测试通过：causal mask、头映射、容量比例和缓存合同均成立。
